# Experiment: Model Layer Inspection

Objective:
- Inspect the pipeline's current stage-end layer mapping for a selected model.
- See candidate module names from `named_modules()` before choosing a decoding layer.


In [ ]:
from __future__ import annotations

import pandas as pd
import timm

from epgabor_v1_pipeline import list_available_layers


## Choose a model

Set `model_name` and whether you want pretrained weights. For layer inspection, pretrained weights are usually not necessary.


In [ ]:
model_name = 'resnet50'
pretrained = False

model = timm.create_model(model_name, pretrained=pretrained)
model.eval()
type(model).__name__


## Current pipeline layer mapping

These are the stage-end layer names currently supported by the decoding pipeline for the selected model.


In [ ]:
layer_df = list_available_layers(model_name)
layer_df


## Candidate module names from `named_modules()`

Use this table to inspect the model structure and identify candidate module paths if you want to revise the stage-end mapping later.


In [ ]:
module_rows = []
for name, module in model.named_modules():
    module_rows.append({
        "module_path": name or "<root>",
        "module_type": module.__class__.__name__,
    })

modules_df = pd.DataFrame(module_rows)
modules_df.head(40)


## Test whether a module path resolves

Use this helper to check whether a candidate module path can be addressed by `model.get_submodule(...)`.


In [ ]:
def inspect_module(module_path: str):
    try:
        module = model.get_submodule(module_path)
    except AttributeError as exc:
        return {
            "module_path": module_path,
            "resolved": False,
            "error": str(exc),
        }

    return {
        "module_path": module_path,
        "resolved": True,
        "module_type": module.__class__.__name__,
    }

inspect_module('layer3') if model_name == 'resnet50' else inspect_module(layer_df.loc[0, 'module_path'])
